<center><font size=10>Generative AI for Business Applications</center></font>
<center><font size=6>Transformers for Text Generation - Week 1</center></font>

<center><p float="center">
  <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRqKnrZC15krW2aqfQM75eT597LYGeF1A2KlizXPkvBrw&s=10" width=780></a>
<center><font size=6>Smart Research Assistant</center></font>

# Problem Statement

## Business Context

* Large amounts of technical information are time-consuming to read.
* Helps employees quickly understand important information.
* Provides summaries and answers questions from articles.
* Reduces research time and improves productivity.



##  Objective

* Develop a Smart Research Assistant for technical articles.
* Summarize lengthy articles and provide relevant answers.
* Make complex information easier to understand.
* Help employees access knowledge faster.

## Data Description

The dataset has two columns:

* **Title**: A brief headline summarizing the main topic of the article.
* **Article**: Full unstructured text containing the detailed content.

# Importing Necessary Libraries

We install specific tested library versions to ensure compatibility and avoid errors during development.


In [ ]:
!pip install \
    numpy==1.26.4 \
    transformers==4.53.1 \
    pandas==2.2.2 \
    streamlit==1.47.0 \
    torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.3 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.2
    Uninstalling packaging-26.2:
      Successfully uninstalled packaging-26.2
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully unins

Note:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer , AutoModelForSeq2SeqLM
import warnings

warnings.filterwarnings("ignore")

# Loading the Data

We’ll use the Pandas library to load the data. Pandas makes it easy to work with tables of data and take a quick look at what’s inside.

Let’s load the dataset and see what it looks like.

In [ ]:
 # Mount the Google Drive

 from google.colab import drive
 drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load the CSV file named "Articles" and store it in the variable data.

data = pd.read_csv('/content/drive/MyDrive/Dataset/GenAIDataset/Articles.csv')

# Data Overview

In [ ]:
# Display the number of rows and columns in the `data`.

data.shape

(10, 2)

The dataset consists of 10 rows and 2 columns

In [ ]:
# Display the first 5 rows of the

data.head(5)


,Title,Article
0,The Information Overload Dilemma: Why Tech Com...,"In today’s hyper-connected digital age, the te..."
1,From Raw Data to Strategic Insight: How NLP is...,In an environment where data is being generate...
2,Building a Competitive Intelligence Platform: ...,As tech companies seek to maintain a strategic...
3,Tech Trends 2025: The Role of Summarization Mo...,"In the tech industry, innovation is not a peri..."
4,The Future of Competitive Intelligence is Auto...,The nature of competitive intelligence is unde...


In [ ]:
# Display the names, data types, and number of entries in the columns of the data.

data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Title    10 non-null     object
 1   Article  10 non-null     object
dtypes: object(2)
memory usage: 292.0+ bytes


# Model Loading

We use the **Hugging Face Transformers** library, a popular open-source platform that provides access to thousands of pre-trained LLM models. It simplifies the process of working with complex transformer architectures by offering:

* A unified API for models like FLAN T5, Tiny Llama, Mistral, etc.
* Easy model loading from the Hugging Face Hub

You can explore thousands of open-source models and datasets on the [Hugging Face Model Hub](https://huggingface.co/)


In this case study, we experiment with the following pre-trained models:

1. `google/flan-t5-large`
2. `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

## Setting up Hugging Face token

Follow these steps to securely set up your Hugging Face token

**Step 1: Create Your Token on Hugging Face**

1. Visit: [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Click on **"New token"**
3. Give it a name (e.g., `colab_token`)
4. Set the role as **"Read"**
5. Click **"Create token"**
6. **Copy** the token that gets generated

**Step 2: Store Your Token Securely in config file**

Paste your token securely in the config file provided.

**Step 2: Run the code below to store your HF_TOKEN securely in os enviroment:**

In [ ]:
import json
import os

# Load from your config.json (update the path if needed)
with open("/content/drive/MyDrive/Dataset/GenAIDataset/config.json", "r") as f:
    config = json.load(f)

# Extract the token
hf_token = config.get("HF_TOKEN")

# Store in Colab's environment variables
os.environ["HF_TOKEN"] = hf_token



You’re now ready to use Hugging Face models!

## FLAN-T5 Large

There are different ways to load AI models, and each model may have its own specific method depending on how it is built and shared. So it's best to follow the official Hugging Face documentation to load it correctly.

You can search for the model on Hugging Face or use this link:
🔗 [https://huggingface.co/google/flan-t5-large](https://huggingface.co/google/flan-t5-large)

Once you're on the model page:

* Go to the **"Use this model"** section, and from the dropdown, choose **"Use in Colab"** or **"Use in Kaggle"** to see example code for loading the model.

* This will open ready-made code examples that show you how to load and run the model
* You can follow the instructions directly to avoid errors

Checking the documentation is always a good practice to make sure you are using the model in the recommended way.


In our case, we’re using the **FLAN-T5 large** and the **TinyLlama** from **Hugging Face**.

**FLAN-T5 (google/flan-t5-large)**

1. An **encoder-decoder** model designed for sequence-to-sequence tasks like summarization and question answering.
2. Trained using **instruction tuning**, allowing it to follow specific input prompts and generate structured, task-specific outputs.
3. FLAN-T5 Large supports a context window of up to 512 tokens (only input)


***Prompt***:

<font size=3 color="#4682B4"><b> Load the `google/flan-t5-large` from hugging face
</font>

In [ ]:
import os
os.environ["HF_TOKEN"] = ""

from huggingface_hub import login
login("")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"
tokenizer_flant5 = AutoTokenizer.from_pretrained(model_name, token=os.environ["HF_TOKEN"])
model_flant5 = AutoModelForSeq2SeqLM.from_pretrained(model_name, token=os.environ["HF_TOKEN"])


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## TinyLlama

**TinyLlama (TinyLlama-1.1B-Chat-v1.0)**

1. It is a **decoder-only model**, trained using causal language modeling.
2. It generates fluent and conversational text, optimized for speed and efficiency on smaller hardware.
3. TinyLlama is a **causal language model (CLM)**, meaning it is trained to predict the next word given previous words only.
4. TinyLlama supports a context window of up to 2,048 tokens, which includes both input and output combined.



***Prompt***:

<font size=3 color="#4682B4"><b> Load the `TinyLlama/TinyLlama-1.1B-Chat-v1.0` from hugging face
</font>

In [ ]:
model_name_tinyllama = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer_tinyllama = AutoTokenizer.from_pretrained(model_name_tinyllama)
model_tinyllama = AutoModelForCausalLM.from_pretrained(model_name_tinyllama)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

# Text Summarization

To begin the summarization process, we extract two articles from our dataset and store it in a variable called sample_article. This article will be used as the input for testing the model's output.



***Prompt***:

<font size=3 color="#4682B4"><b> Select first article from the dataset and store it in a variable named sample_article_1.
</font>

In [ ]:
sample_article_1 = data['Article'][0]
sample_article_1

'In today’s hyper-connected digital age, the tech industry is evolving at a rapid pace, with a constant influx of information from news articles, research papers, press releases, social media, and analyst reports. While this stream of data holds immense strategic value, it also presents a serious challenge: information overload. Tech companies are struggling to manage the volume, relevance, and timeliness of this information, leading to delays in product development, missed market signals, and ineffective strategic planning. The problem is compounded by the fragmented nature of sources and the difficulty in distinguishing valuable insights from repetitive or shallow content. Traditionally, companies have relied on analysts to manually scan, read, and summarize this information, a process that is not only slow and resource-intensive but also prone to human bias and oversight. This is where Natural Language Processing (NLP) emerges as a transformative solution. With state-of-the-art tran

***Prompt***:

<font size=3 color="#4682B4"><b> Select sixth article  from the dataset and store it in a variable named sample_article_2.
</font>

In [ ]:
sample_article_2 = data['Article'][5]
sample_article_2

'For many decision-makers across tech companies—whether in executive leadership, product strategy, or innovation management—Natural Language Processing (NLP) has emerged as one of the most transformative yet misunderstood technologies in modern business intelligence. At its core, NLP is the ability of machines to read, understand, and generate human language, enabling companies to extract meaning from massive volumes of unstructured text data. Unlike traditional analytics that work best on numbers and structured databases, NLP specializes in processing content that humans create every day—news stories, product reviews, market reports, social media posts, and internal documentation. This makes it uniquely suited to the kinds of insights that matter most to business leaders: what customers are saying, what competitors are announcing, how sentiment is shifting, and where market conversations are headed. Modern NLP is powered by large language models (LLMs) like GPT-4, BERT, RoBERTa, and T

## FLAN-T5 Large

***Prompt***:

<font size=3 color="#4682B4"><b>Create a function that takes an article as input and returns its summary using the FLAN-T5 model.

</font>

In [ ]:
def summarize_flant5(article):
    input_text = f"Summarize the following article, ensuring all key details and insights are captured clearly and concisely in the summary : {article}"
    inputs = tokenizer_flant5(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = model_flant5.generate(inputs["input_ids"], max_length=150, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)
    summary = tokenizer_flant5.decode(outputs[0], skip_special_tokens=True)
    return summary

### Article 1

In [ ]:
output=summarize_flant5(sample_article_1)
print(output)

Using Natural Language Processing (NLP) to read smartly will help organizations manage information overload and turn it into a powerful asset for innovation and long-term competitiveness. Using NLP to read smartly will help organizations manage information overload and turn it into a powerful asset for innovation and long-term competitiveness.


**Model Response**

> **Article** : In today’s hyper-connected digital age, the tech industry is evolving at a rapid pace, with a constant influx of information from news articles, research papers, press releases, social media, and analyst reports. While this stream of data holds immense strategic value, it also presents a serious challenge: information overload. Tech companies are struggling to manage the volume, relevance, and timeliness of this information, leading to delays in product development, missed market signals, and ineffective strategic planning. The problem is compounded by the fragmented nature of sources and the difficulty in distinguishing valuable insights from repetitive or shallow content. Traditionally, companies have relied on analysts to manually scan, read, and summarize this information, a process that is not only slow and resource-intensive but also prone to human bias and oversight. This is where Natural Language Processing (NLP) emerges as a transformative solution. With state-of-the-art transformer models like BART, T5, and GPT, companies can now automatically summarize long-form content, extract key entities and keywords, detect sentiment, and cluster related topics for pattern recognition. These capabilities enable faster, more accurate synthesis of competitive and market intelligence. By integrating such NLP tools into a centralized platform, companies can automate data ingestion, filtering, summarization, and alerting, ensuring that only the most relevant insights reach decision-makers in real time. This shift from manual to AI-assisted intelligence allows organizations to be more agile, make faster decisions, and gain a strategic edge. Real-world examples already show companies leveraging NLP to track competitor activity, summarize industry trends, and deliver tailored updates to different departments, significantly improving internal communication and responsiveness. Whether building an in-house platform or adopting third-party tools, the key lies in customizing the NLP system to align with strategic goals, data privacy needs, and the pace of market evolution. As information volumes continue to rise, the real advantage will belong to organizations that stop trying to read everything and start using AI to read smartly. In doing so, they will not only manage information overload but turn it into a powerful asset for innovation and long-term competitiveness.

> **Summary** : Using Natural Language Processing (NLP) to read smartly will help organizations manage information overload and turn it into a powerful asset for innovation and long-term competitiveness. Using NLP to read smartly will help organizations manage information overload and turn it into a powerful asset for innovation and long-term competitiveness.

### Article 2

In [ ]:
output=summarize_flant5(sample_article_2)
print(output)

Outmaneu is a technology that enables companies to extract meaningful insights from unstructured text data. Outmaneu is a technology that enables companies to extract meaningful insights from unstructured text data. Outmaneu is a technology that enables companies to extract meaningful insights from unstructured text data.


**Model Response**

> **Article** : For many decision-makers across tech companies—whether in executive leadership, product strategy, or innovation management—Natural Language Processing (NLP) has emerged as one of the most transformative yet misunderstood technologies in modern business intelligence. At its core, NLP is the ability of machines to read, understand, and generate human language, enabling companies to extract meaning from massive volumes of unstructured text data. Unlike traditional analytics that work best on numbers and structured databases, NLP specializes in processing content that humans create every day—news stories, product reviews, market reports, social media posts, and internal documentation. This makes it uniquely suited to the kinds of insights that matter most to business leaders: what customers are saying, what competitors are announcing, how sentiment is shifting, and where market conversations are headed. Modern NLP is powered by large language models (LLMs) like GPT-4, BERT, RoBERTa, and T5, which use deep learning and transformer architectures to understand context, intent, and nuance. These models can automatically summarize 20-page reports into five-sentence briefs, extract key information such as competitor names, product launches, and pricing changes, and even classify documents by theme, urgency, or relevance. For decision-makers, this means less time reading and more time acting—receiving focused updates rather than digging through dozens of scattered documents or emails. For example, a CEO might get a daily digest highlighting any strategic moves made by top five competitors, while a CMO could track which messaging trends are gaining traction in recent tech product launches. NLP tools can also scan earnings calls or investor briefings to detect hidden concerns, confidence levels, or repeated talking points that signal a shift in company direction. What makes NLP especially powerful is its scalability and adaptability—once set up, it can monitor thousands of sources simultaneously, in multiple languages, across global markets. And because these tools are trainable, they can learn what matters most to a specific organization, tailoring summaries and filters accordingly. Importantly, NLP doesn’t replace human judgment—it enhances it by delivering sharper, faster, and more relevant inputs. For decision-makers looking to navigate uncertainty, outmaneuver competitors, and capitalize on early signals, NLP offers a strategic capability that goes far beyond automation—it provides clarity, context, and competitive advantage in an increasingly noisy world.

> **Summary** : Outmaneu is a technology that enables companies to extract meaningful insights from unstructured text data. Outmaneu is a technology that enables companies to extract meaningful insights from unstructured text data. Outmaneu is a technology that enables companies to extract meaningful insights from unstructured text data.


### Model Result

Both the summaries appear repetitive because FLAN-T5 has a maximum input limit of 512 tokens, whereas the articles being provided exceed that limit. As a result, much of the input gets truncated, leading to incomplete context and repetitive outputs.

**Limitation of FLAN-T5**

1. **Older Architecture**
   FLAN-T5 is an older encoder-decoder model and may underperform compared to newer models fine-tuned for nuanced generation.

2. **Fewer Parameters**
   FLAN-T5-large has fewer parameters and less training data, limiting its ability to handle complex or detailed content.

3. **Context Limit of 512 Tokens**
   The model can only process up to 512 tokens at once. Inputs beyond this (e.g., 1000+ word articles) may be truncated, leading to incomplete summaries.


To handle the context window limitation, we can use larger variants like **FLAN-T5 XL** or **FLAN-T5 XXL**, which support significantly higher token limits (up to 2048 tokens), allowing for more complete input processing and improved summary quality.


Having explored the limitations of FLAN-T5, let’s now test the TinyLlama model, which supports a higher token limit of 2048, allowing better handling of longer article inputs.

Let’s see how it performs on the same article using a summarization or question-answering approach.


## Tiny Llama

***Prompt***:

<font size=3 color="#4682B4"><b> Create a function that takes an article as input and returns its summary using the Tiny Llama model.
</font>

In [ ]:
def summarize_tinyllama(article):
    # For causal models like TinyLlama, summarization isn't a direct task like with encoder-decoder models.
    # We can prompt it to continue a summary.
    prompt="Summarize the following article clearly and concisely:"
    input_text = f"{prompt}\n{article}\nSummary:"
    inputs = tokenizer_tinyllama(input_text, return_tensors="pt", max_length=1024, truncation=True)

    # Generate tokens - the model will try to complete the input prompt.
    # We need to adjust generation parameters for open-ended generation.
    # max_new_tokens controls how much new text is generated after the prompt.
    outputs = model_tinyllama.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=500,  # Generate up to 300 new tokens for the summary
        do_sample=True,     # Don't sample, use greedy decoding
        temperature=0.7,
        min_new_tokens=150,
        top_p=0.9,
        pad_token_id=tokenizer_tinyllama.eos_token_id, # Pad with EOS token if needed
    )

    # Decode the entire output sequence.
    generated_text = tokenizer_tinyllama.decode(outputs[0], skip_special_tokens=True)

    # The generated text will include the original prompt. We need to extract the summary part.
    # This is a simple approach, more sophisticated parsing might be needed depending on prompt and output.
    summary_start_index = generated_text.find("Summary:") + len("Summary:")
    summary = generated_text[summary_start_index:].strip()

    return summary

### Article 1

In [ ]:
output=summarize_tinyllama(sample_article_1)
print(output)

The article discusses the challenges faced by tech companies due to information overload and how Natural Language Processing (NLP) can help manage this issue. It provides real-world examples of how NLP is used in the tech industry to streamline data analysis and communication. The article highlights the advantages of using AI for information management and how it can help companies innovate and stay ahead in a rapidly evolving industry. Overall, the article provides a valuable insight into the use of NLP in the tech industry and its potential impact on businesses.
Keywords: Information Overload, Natural Language Processing, Tech Industry, Analysts, Strategic Planning, Competitive Intelligence, AI, Innovation, Long-term Competitiveness


**Model Response**

> **Article** : In today’s hyper-connected digital age, the tech industry is evolving at a rapid pace, with a constant influx of information from news articles, research papers, press releases, social media, and analyst reports. While this stream of data holds immense strategic value, it also presents a serious challenge: information overload. Tech companies are struggling to manage the volume, relevance, and timeliness of this information, leading to delays in product development, missed market signals, and ineffective strategic planning. The problem is compounded by the fragmented nature of sources and the difficulty in distinguishing valuable insights from repetitive or shallow content. Traditionally, companies have relied on analysts to manually scan, read, and summarize this information, a process that is not only slow and resource-intensive but also prone to human bias and oversight. This is where Natural Language Processing (NLP) emerges as a transformative solution. With state-of-the-art transformer models like BART, T5, and GPT, companies can now automatically summarize long-form content, extract key entities and keywords, detect sentiment, and cluster related topics for pattern recognition. These capabilities enable faster, more accurate synthesis of competitive and market intelligence. By integrating such NLP tools into a centralized platform, companies can automate data ingestion, filtering, summarization, and alerting, ensuring that only the most relevant insights reach decision-makers in real time. This shift from manual to AI-assisted intelligence allows organizations to be more agile, make faster decisions, and gain a strategic edge. Real-world examples already show companies leveraging NLP to track competitor activity, summarize industry trends, and deliver tailored updates to different departments, significantly improving internal communication and responsiveness. Whether building an in-house platform or adopting third-party tools, the key lies in customizing the NLP system to align with strategic goals, data privacy needs, and the pace of market evolution. As information volumes continue to rise, the real advantage will belong to organizations that stop trying to read everything and start using AI to read smartly. In doing so, they will not only manage information overload but turn it into a powerful asset for innovation and long-term competitiveness.

> **Summary** : The article highlights the challenges that the tech industry faces in managing information overload. It explains how NLP technology can help companies automate data ingestion, filtering, summarization, and alerting. The article also provides real-world examples of how AI is used to track competitor activity, summarize industry trends, and deliver tailored updates to different departments. The article emphasizes the importance of customizing the NLP system to align with strategic goals, data privacy needs, and the pace of market evolution. The article suggests that organizations that stop trying to read everything and start using AI to read smartly will not only manage information overload but also turn it into a powerful asset for innovation and long-term competitiveness.

### Article 2

In [ ]:
output=summarize_tinyllama(sample_article_2)
print(output)

Natural Language Processing (NLP) is a transformative yet misunderstood technology in modern business intelligence. At its core, NLP enables companies to extract meaning from massive volumes of unstructured text data. Unlike traditional analytics, which work best on numbers and structured databases, NLP specializes in processing content that humans create every day—news stories, product reviews, market reports, social media posts, and internal documentation. This makes it uniquely suited to the kinds of insights that matter most to business leaders: what customers are saying, what competitors are announcing, how sentiment is shifting, and where market conversations are headed. Modern NLP is powered by large language models (LLMs) like GPT-4, BERT, RoBERTa, and T5, which use deep learning and transformer architectures to understand context, intent, and nuance. These models can automatically summarize 20-page reports into five-sentence briefs, extract key information such as competitor n

**Model Response**

> **Article** : For many decision-makers across tech companies—whether in executive leadership, product strategy, or innovation management—Natural Language Processing (NLP) has emerged as one of the most transformative yet misunderstood technologies in modern business intelligence. At its core, NLP is the ability of machines to read, understand, and generate human language, enabling companies to extract meaning from massive volumes of unstructured text data. Unlike traditional analytics that work best on numbers and structured databases, NLP specializes in processing content that humans create every day—news stories, product reviews, market reports, social media posts, and internal documentation. This makes it uniquely suited to the kinds of insights that matter most to business leaders: what customers are saying, what competitors are announcing, how sentiment is shifting, and where market conversations are headed. Modern NLP is powered by large language models (LLMs) like GPT-4, BERT, RoBERTa, and T5, which use deep learning and transformer architectures to understand context, intent, and nuance. These models can automatically summarize 20-page reports into five-sentence briefs, extract key information such as competitor names, product launches, and pricing changes, and even classify documents by theme, urgency, or relevance. For decision-makers, this means less time reading and more time acting—receiving focused updates rather than digging through dozens of scattered documents or emails. For example, a CEO might get a daily digest highlighting any strategic moves made by top five competitors, while a CMO could track which messaging trends are gaining traction in recent tech product launches. NLP tools can also scan earnings calls or investor briefings to detect hidden concerns, confidence levels, or repeated talking points that signal a shift in company direction. What makes NLP especially powerful is its scalability and adaptability—once set up, it can monitor thousands of sources simultaneously, in multiple languages, across global markets. And because these tools are trainable, they can learn what matters most to a specific organization, tailoring summaries and filters accordingly. Importantly, NLP doesn’t replace human judgment—it enhances it by delivering sharper, faster, and more relevant inputs. For decision-makers looking to navigate uncertainty, outmaneuver competitors, and capitalize on early signals, NLP offers a strategic capability that goes far beyond automation—it provides clarity, context, and competitive advantage in an increasingly noisy world.

> **Summary** : The article discusses the transformative power of Natural Language Processing (NLP) in modern business intelligence. It highlights how NLP has emerged as a critical tool for decision-makers across tech companies, enabling them to extract meaning from massive volumes of unstructured text data. The article explains how NLP is powered by large language models (LLMs) like GPT-4, BERT, RoBERTa, and T5, which use deep learning and transformer architectures to understand context, intent, and nuance. The article also highlights the importance of scalability and adaptability in NLP, as these tools can monitor thousands of sources simultaneously, in multiple languages, and adapt to different organizations' needs. The article concludes by emphasizing the strategic capability of NLP, which provides clarity, context, and competitive advantage in an increasingly noisy world.

### Model Result

- We observe that TinyLlama is capable of generating high-quality summaries that effectively capture the key points and important details from the original content.
- The summaries are concise yet informative, demonstrating the model’s ability to retain context and highlight the most relevant information.
- This makes TinyLlama a useful tool for tasks that require quick understanding of lengthy texts.

# Question-Answering

## FLAN-T5 Large

We observed that FLAN-T5 struggles to handle longer text inputs effectively, which is why we are not using it to generate answers in this case.

## Tiny Llama

***Prompt***:

<font size=3 color="#4682B4"><b> Create a function that takes an article and a question as input, and returns the answer generated by the TinyLlama model.
</font>

In [ ]:
def answer_question_tinyllama(article, question):
    # Formulate the prompt to guide the TinyLlama model to answer the question based on the article.
    # We ask the model to act as an AI answering a question based on the provided text.
    input_text = f"From this Article: {article}\n\n Answer the below Question: {question}\n\nAnswer:"

    # Tokenize the input text
    # Truncate if the combined article and question is too long
    inputs = tokenizer_tinyllama(input_text, return_tensors="pt", max_length=1024, truncation=True ,padding=True)

    # Generate the answer using the model.
    # We use generate with parameters suitable for generating a concise answer.
    outputs = model_tinyllama.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=500,  # Generate up to 100 new tokens for the answer
        do_sample=True,      # Use sampling to potentially get more varied answers
        temperature=0.7,     # Control randomness
        top_p=0.9,           # Nucleus sampling
        pad_token_id=tokenizer_tinyllama.eos_token_id, # Pad with EOS token if needed
    )

    # Decode the generated sequence
    generated_text = tokenizer_tinyllama.decode(outputs[0], skip_special_tokens=True)

    # The generated text will include the original prompt. We need to extract the answer part.
    # This is a simple approach, more sophisticated parsing might be needed depending on prompt and output.
    answer_start_index = generated_text.find("Answer:") + len("Answer:")
    answer = generated_text[answer_start_index:].strip()

    # Basic cleanup: remove potential repetition of the question or prompt in the answer
    if answer.startswith(question):
        answer = answer[len(question):].strip()

    return answer


### Article 1

#### Question : What are the models that are trending currently?

In [ ]:
# Example Usage:
sample_question = "What are the models that are trending currently?"
answer = answer_question_tinyllama(sample_article_1, sample_question)
print(f"Answer: {answer}")

Answer: NLP models like BART, T5, and GPT are trending models that are currently in use by tech companies to automate data ingestion, filtering, summarization, and alerting. These models have the potential to revolutionize the way companies manage information overload by automating the data ingestion process.


> **Article** : In today’s hyper-connected digital age, the tech industry is evolving at a rapid pace, with a constant influx of information from news articles, research papers, press releases, social media, and analyst reports. While this stream of data holds immense strategic value, it also presents a serious challenge: information overload. Tech companies are struggling to manage the volume, relevance, and timeliness of this information, leading to delays in product development, missed market signals, and ineffective strategic planning. The problem is compounded by the fragmented nature of sources and the difficulty in distinguishing valuable insights from repetitive or shallow content. Traditionally, companies have relied on analysts to manually scan, read, and summarize this information, a process that is not only slow and resource-intensive but also prone to human bias and oversight. This is where Natural Language Processing (NLP) emerges as a transformative solution. With state-of-the-art transformer models like BART, T5, and GPT, companies can now automatically summarize long-form content, extract key entities and keywords, detect sentiment, and cluster related topics for pattern recognition. These capabilities enable faster, more accurate synthesis of competitive and market intelligence. By integrating such NLP tools into a centralized platform, companies can automate data ingestion, filtering, summarization, and alerting, ensuring that only the most relevant insights reach decision-makers in real time. This shift from manual to AI-assisted intelligence allows organizations to be more agile, make faster decisions, and gain a strategic edge. Real-world examples already show companies leveraging NLP to track competitor activity, summarize industry trends, and deliver tailored updates to different departments, significantly improving internal communication and responsiveness. Whether building an in-house platform or adopting third-party tools, the key lies in customizing the NLP system to align with strategic goals, data privacy needs, and the pace of market evolution. As information volumes continue to rise, the real advantage will belong to organizations that stop trying to read everything and start using AI to read smartly. In doing so, they will not only manage information overload but turn it into a powerful asset for innovation and long-term competitiveness.

> Question : What are the models that are trending currently?

> Answer : The models that are trending currently are BART, T5, and GPT.

#### Question : Why there is a need of Text Summarization?

In [ ]:
# Example Usage:
sample_question = "Why there is a need of Text Summarization?"
answer = answer_question_tinyllama(sample_article_1, sample_question)
print(f"Answer: {answer}")

Answer: Text summarization is a process of condensing long text into a concise and readable format. This is done by identifying and extracting the most important and relevant information from the text and presenting it in a structured manner. It helps in improving the understanding of the content, making it more accessible and useful for the target audience. With the increasing volume and complexity of text-based data, text summarization has become a critical process in modern businesses. It helps in improving data analysis, decision-making, and communication. Text summarization can also help in reducing the workload of analysts and other stakeholders by providing a consolidated and organized view of the information. In summary, text summarization is a crucial process that helps businesses in improving their understanding of complex information, reducing workload, and improving communication.


> **Article** : In today’s hyper-connected digital age, the tech industry is evolving at a rapid pace, with a constant influx of information from news articles, research papers, press releases, social media, and analyst reports. While this stream of data holds immense strategic value, it also presents a serious challenge: information overload. Tech companies are struggling to manage the volume, relevance, and timeliness of this information, leading to delays in product development, missed market signals, and ineffective strategic planning. The problem is compounded by the fragmented nature of sources and the difficulty in distinguishing valuable insights from repetitive or shallow content. Traditionally, companies have relied on analysts to manually scan, read, and summarize this information, a process that is not only slow and resource-intensive but also prone to human bias and oversight. This is where Natural Language Processing (NLP) emerges as a transformative solution. With state-of-the-art transformer models like BART, T5, and GPT, companies can now automatically summarize long-form content, extract key entities and keywords, detect sentiment, and cluster related topics for pattern recognition. These capabilities enable faster, more accurate synthesis of competitive and market intelligence. By integrating such NLP tools into a centralized platform, companies can automate data ingestion, filtering, summarization, and alerting, ensuring that only the most relevant insights reach decision-makers in real time. This shift from manual to AI-assisted intelligence allows organizations to be more agile, make faster decisions, and gain a strategic edge. Real-world examples already show companies leveraging NLP to track competitor activity, summarize industry trends, and deliver tailored updates to different departments, significantly improving internal communication and responsiveness. Whether building an in-house platform or adopting third-party tools, the key lies in customizing the NLP system to align with strategic goals, data privacy needs, and the pace of market evolution. As information volumes continue to rise, the real advantage will belong to organizations that stop trying to read everything and start using AI to read smartly. In doing so, they will not only manage information overload but turn it into a powerful asset for innovation and long-term competitiveness.

> Question : Why there is a need of Text Summarization?

> Answer : Text summarization is a process of extracting key ideas, concepts, and information from long-form textual content. It is a crucial step in the process of information retrieval, analysis, and decision-making. The need for text summarization arises due to the large volume of unstructured data generated by various sources such as news articles, research papers, press releases, social media, and analyst reports. This data is often difficult to analyze and extract relevant insights from. Traditionally, organizations have relied on manual summarization methods such as scanning, reading, and summarizing content. However, this process is time-consuming, resource-intensive, and prone to human bias and oversight. With the advent of natural language processing (NLP) technologies, organizations can now automate text summarization tasks. These technologies enable faster, more accurate, and more efficient summarization of competitive and market intelligence. By integrating NLP into a centralized platform, organizations can automate data ingestion, filtering, summarization, and alerting, ensuring that only the most relevant insights reach decision-makers in real-time. This shift from manual to AI-assisted intelligence allows organizations to be more agile, make faster decisions, and gain a strategic edge.

### Article 2

#### Question : How is NLP different from traditional analytics in terms of data types it handles?

In [ ]:
# Example Usage:
sample_question = "How is NLP different from traditional analytics in terms of data types it handles?"
answer = answer_question_tinyllama(sample_article_2, sample_question)
print(f"Answer: {answer}")

Answer: NLP is different from traditional analytics in terms of data types it handles. NLP is a highly specialized technology that processes unstructured text data, enabling companies to extract meaning from vast amounts of unstructured information. Traditional analytics, on the other hand, works best on numbers and structured databases.


> **Article** : For many decision-makers across tech companies—whether in executive leadership, product strategy, or innovation management—Natural Language Processing (NLP) has emerged as one of the most transformative yet misunderstood technologies in modern business intelligence. At its core, NLP is the ability of machines to read, understand, and generate human language, enabling companies to extract meaning from massive volumes of unstructured text data. Unlike traditional analytics that work best on numbers and structured databases, NLP specializes in processing content that humans create every day—news stories, product reviews, market reports, social media posts, and internal documentation. This makes it uniquely suited to the kinds of insights that matter most to business leaders: what customers are saying, what competitors are announcing, how sentiment is shifting, and where market conversations are headed. Modern NLP is powered by large language models (LLMs) like GPT-4, BERT, RoBERTa, and T5, which use deep learning and transformer architectures to understand context, intent, and nuance. These models can automatically summarize 20-page reports into five-sentence briefs, extract key information such as competitor names, product launches, and pricing changes, and even classify documents by theme, urgency, or relevance. For decision-makers, this means less time reading and more time acting—receiving focused updates rather than digging through dozens of scattered documents or emails. For example, a CEO might get a daily digest highlighting any strategic moves made by top five competitors, while a CMO could track which messaging trends are gaining traction in recent tech product launches. NLP tools can also scan earnings calls or investor briefings to detect hidden concerns, confidence levels, or repeated talking points that signal a shift in company direction. What makes NLP especially powerful is its scalability and adaptability—once set up, it can monitor thousands of sources simultaneously, in multiple languages, across global markets. And because these tools are trainable, they can learn what matters most to a specific organization, tailoring summaries and filters accordingly. Importantly, NLP doesn’t replace human judgment—it enhances it by delivering sharper, faster, and more relevant inputs. For decision-makers looking to navigate uncertainty, outmaneuver competitors, and capitalize on early signals, NLP offers a strategic capability that goes far beyond automation—it provides clarity, context, and competitive advantage in an increasingly noisy world.

> Question : How is NLP different from traditional analytics in terms of data types it handles?

> Answer : Natural Language Processing (NLP) is a type of artificial intelligence (AI) that specializes in processing text data. Unlike traditional analytics, which focuses on numbers and structured databases, NLP uses natural language to analyze human-generated content. NLP tools can read, understand, and generate human language, enabling companies to extract meaning from massive volumes of unstructured text data. Unlike traditional analytics that work best on numbers and structured databases, NLP specializes in processing content that humans create every day—news stories, product reviews, market reports, social media posts, and internal documentation. This makes it uniquely suited to the kinds of insights that matter most to business leaders: what customers are saying, what competitors are announcing, how sentiment is shifting, and where market conversations are headed.

#### Question : Why is NLP seen as a strategic tool rather than just automation?

In [ ]:
# Example Usage:
sample_question = "Why is NLP seen as a strategic tool rather than just automation?"
answer = answer_question_tinyllama(sample_article_2, sample_question)
print(f"Answer: {answer}")

Answer: NLP is seen as a strategic tool rather than just automation because it offers a unique ability to process unstructured text data and deliver insights that traditional analytics cannot. Unlike traditional analytics that work best on numbers and structured databases, NLP specializes in processing content that humans create every day—news stories, product reviews, market reports, social media posts, and internal documentation. This makes it uniquely suited to the kinds of insights that matter most to business leaders: what customers are saying, what competitors are announcing, how sentiment is shifting, and where market conversations are headed.


> **Article** : For many decision-makers across tech companies—whether in executive leadership, product strategy, or innovation management—Natural Language Processing (NLP) has emerged as one of the most transformative yet misunderstood technologies in modern business intelligence. At its core, NLP is the ability of machines to read, understand, and generate human language, enabling companies to extract meaning from massive volumes of unstructured text data. Unlike traditional analytics that work best on numbers and structured databases, NLP specializes in processing content that humans create every day—news stories, product reviews, market reports, social media posts, and internal documentation. This makes it uniquely suited to the kinds of insights that matter most to business leaders: what customers are saying, what competitors are announcing, how sentiment is shifting, and where market conversations are headed. Modern NLP is powered by large language models (LLMs) like GPT-4, BERT, RoBERTa, and T5, which use deep learning and transformer architectures to understand context, intent, and nuance. These models can automatically summarize 20-page reports into five-sentence briefs, extract key information such as competitor names, product launches, and pricing changes, and even classify documents by theme, urgency, or relevance. For decision-makers, this means less time reading and more time acting—receiving focused updates rather than digging through dozens of scattered documents or emails. For example, a CEO might get a daily digest highlighting any strategic moves made by top five competitors, while a CMO could track which messaging trends are gaining traction in recent tech product launches. NLP tools can also scan earnings calls or investor briefings to detect hidden concerns, confidence levels, or repeated talking points that signal a shift in company direction. What makes NLP especially powerful is its scalability and adaptability—once set up, it can monitor thousands of sources simultaneously, in multiple languages, across global markets. And because these tools are trainable, they can learn what matters most to a specific organization, tailoring summaries and filters accordingly. Importantly, NLP doesn’t replace human judgment—it enhances it by delivering sharper, faster, and more relevant inputs. For decision-makers looking to navigate uncertainty, outmaneuver competitors, and capitalize on early signals, NLP offers a strategic capability that goes far beyond automation—it provides clarity, context, and competitive advantage in an increasingly noisy world.

> Question : Why is NLP seen as a strategic tool rather than just automation?

> Answer : NLP is seen as a strategic tool rather than just automation because it offers a way to gather, analyze, and act on vast amounts of unstructured text data. NLP specializes in processing content that humans create every day, enabling companies to extract meaning from it. This makes it uniquely suited to the kinds of insights that matter most to business leaders: what customers are saying, what competitors are announcing, how sentiment is shifting, and where market conversations are headed. NLP tools can scan earnings calls or investor briefings to detect hidden concerns, confidence levels, or repeated talking points that signal a shift in company direction.


### We can see that TinyLlama is generating answers to the questions very effectively.

# Deployment

Now, we deploy this using Hugging Face Spaces with Streamlit.
This is purely for demonstration purposes, so we won’t dive deep into the code here.

## Streamlit on Hugging Face

### app.py

In [ ]:
# %%writefile app.py

# import streamlit as st
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch
# import os
# from huggingface_hub import login


# model_name_tinyllama = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# tokenizer_tinyllama = AutoTokenizer.from_pretrained(model_name_tinyllama)
# model_tinyllama = AutoModelForCausalLM.from_pretrained(model_name_tinyllama,torch_dtype=torch.float32,device_map={"": "cpu"})

# def summarize_tinyllama(article):
#     # For causal models like TinyLlama, summarization isn't a direct task like with encoder-decoder models.
#     # We can prompt it to continue a summary.
#     prompt="Summarize the following article clearly and concisely:"
#     input_text = f"{prompt}\n{article}\nSummary:"
#     inputs = tokenizer_tinyllama(input_text, return_tensors="pt", max_length=1024, truncation=True)

#     # Generate tokens - the model will try to complete the input prompt.
#     # We need to adjust generation parameters for open-ended generation.
#     # max_new_tokens controls how much new text is generated after the prompt.
#     outputs = model_tinyllama.generate(
#         inputs["input_ids"],
#         attention_mask=inputs["attention_mask"],
#         max_new_tokens=500,  # Generate up to 300 new tokens for the summary
#         do_sample=True,     # Don't sample, use greedy decoding
#         temperature=0.7,
#         min_new_tokens=150,
#         top_p=0.9,
#         pad_token_id=tokenizer_tinyllama.eos_token_id, # Pad with EOS token if needed
#     )

#     # Decode the entire output sequence.
#     generated_text = tokenizer_tinyllama.decode(outputs[0], skip_special_tokens=True)


#     # The generated text will include the original prompt. We need to extract the summary part.
#     # This is a simple approach, more sophisticated parsing might be needed depending on prompt and output.
#     summary_start_index = generated_text.find("Summary:") + len("Summary:")
#     summary = generated_text[summary_start_index:].strip()

#     return summary

# def answer_question_tinyllama(article, question):
#     # Formulate the prompt to guide the TinyLlama model to answer the question based on the article.
#     # We ask the model to act as an AI answering a question based on the provided text.
#     input_text = f"From this Article: {article}\n\n Answer the below Question: {question}\n\nAnswer:"

#     # Tokenize the input text
#     # Truncate if the combined article and question is too long
#     inputs = tokenizer_tinyllama(input_text, return_tensors="pt", max_length=1024, truncation=True)

#     # Generate the answer using the model.
#     # We use generate with parameters suitable for generating a concise answer.
#     outputs = model_tinyllama.generate(
#         inputs["input_ids"],
#         attention_mask=inputs["attention_mask"],
#         max_new_tokens=500,  # Generate up to 100 new tokens for the answer
#         do_sample=True,      # Use sampling to potentially get more varied answers
#         temperature=0.7,     # Control randomness
#         top_p=0.9,           # Nucleus sampling
#         pad_token_id=tokenizer_tinyllama.eos_token_id, # Pad with EOS token if needed
#     )
#     # Decode the generated sequence
#     generated_text = tokenizer_tinyllama.decode(outputs[0], skip_special_tokens=True)

#     # The generated text will include the original prompt. We need to extract the answer part.
#     # This is a simple approach, more sophisticated parsing might be needed depending on prompt and output.
#     answer_start_index = generated_text.find("Answer:") + len("Answer:")
#     answer = generated_text[answer_start_index:].strip()

#     # Basic cleanup: remove potential repetition of the question or prompt in the answer
#     if answer.startswith(question):
#         answer = answer[len(question):].strip()

#     return answer


# st.title("Smart Article Insights Generator")
# st.markdown("Summarize an article or ask a question about it.")

# mode = st.radio("Select Mode", ["Summarize", "Answer Question"])

# article_input = st.text_area("Article Text", height=300, placeholder="Paste the article here...")

# question_input = None
# if mode == "Answer Question":
#     question_input = st.text_input("Question", placeholder="Enter your question here...")

# if st.button("Process"):
#     if mode == "Summarize":
#         if article_input:
#             with st.spinner("Generating summary..."):
#                 output = summarize_tinyllama(article_input)
#                 st.subheader("Summary")
#                 st.write(output)
#         else:
#             st.warning("Please provide an article to summarize.")
#     elif mode == "Answer Question":
#         if article_input and question_input:
#             with st.spinner("Generating answer..."):
#                 output = answer_question_tinyllama(article_input, question_input)
#                 st.subheader("Answer")
#                 st.write(output)
#         elif not article_input:
#             st.warning("Please provide an article to answer the question from.")
#         elif not question_input:
#             st.warning("Please provide a question to answer.")


### Docker File

In [ ]:
# %%writefile Dockerfile

# # Use an official Python runtime as a parent image
# FROM python:3.10-slim


# # Set environment variables
# ENV PYTHONUNBUFFERED 1


# # Install system dependencies and git
# RUN apt-get update && apt-get install -y \
#     build-essential \
#     git \
#     && rm -rf /var/lib/apt/lists/*




# # Create a non-root user and set permissions
# RUN useradd -ms /bin/bash appuser
# # Set the working directory in the container
# WORKDIR /home/appuser/app


# # Copy the requirements file and install dependencies
# COPY requirements.txt .
# RUN pip install --upgrade pip && pip install -r requirements.txt


# # Switch to non-root user
# USER appuser


# # Copy the rest of the application code into the container
# COPY --chown=appuser . /home/appuser/app


# # Expose the port that the app runs on
# EXPOSE 8501


# # Command to run the application
# CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]



Writing Dockerfile


### requirements.txt

In [ ]:
# %%writefile requirements.txt
# numpy==1.26.4
# transformers==4.53.2
# pandas==2.2.2
# streamlit==1.47.0
# torch==2.6.0
# accelerate==1.9.0
# huggingface_hub==0.33.4

Writing requirements.txt


## Hugging Face Setup

#### 1. Login to Hugging Face

Go to [Hugging face](https://huggingface.co) and sign up or log in to your account.

#### 2. Create a New Space

   * Navigate to [Hugging face Spaces ](https://huggingface.co/spaces).
   * Click **Create New Space**.
   * Fill in:

     * Name for your Space.
     * Space SDK: Select **Docker**.
     * Choose a Docker template: Select **Streamlit**
     * Visibility: Choose *Public* or *Private*.
   * Click **Create Space**.

#### 3. Setup Hugging Face token




- Open your Hugging Face Space

- Click on the **“Settings”** tab at the top of the Space.

- Scroll down to the **“Repository secrets”** section.

- Click **“Add a new secret”**.

- In the **Name** field, type `HF_TOKEN`

- In the **Secret** field, paste your **HF token** (which you previously generated).

- Click **“Add secret”** to save it.


#### 4. Upload Your Files




   * In the new Space, click the **Files** tab.
   * Delete existing requirements.txt , DockerFile
   * Click Contribute then Upload files and add:

     * `app.py`
     * `requirements.txt`
     * `DockerFile`
   * Commit the upload.

#### 5. Build and Launch



   * Hugging Face will automatically detect the `Dockerfile` and start building the container.
   * Wait a few minutes for the build to complete and the app to go live.

#### 6. Access and Test

* Go to the App tab or the Space URL to view and test your running Streamlit app.



# Conclusion

* We developed a Smart Article Insights Generator to tackle information overload in the tech industry using transformer models such as FLAN-T5 (encoder-decoder) and TinyLlama (decoder-only).

* FLAN-T5 faced limitations with long texts, while TinyLlama handled targeted queries well despite its small size and causal nature.

* A Streamlit interface enables users to summarize articles or extract specific insights, supporting better decision-making.


**Future Enhancements**


* Lightweight base models like FLAN-T5 and TinyLlama have limited token capacity, which makes them struggle with long articles or multi-part tasks.
  *Example*: When prompted with “Summarize this long article and also compare it with recent advancements in AI, in 100 words,” the model may skip key points or produce overly generic answers.

* To tackle these issues, effective **prompt engineering** becomes essential. Breaking down complex queries into smaller, focused prompts can help guide the model more accurately and improve the relevance of its responses.


* When dealing with large volumes of articles or documents exceeding token limits, models can’t process all the content at once.
  *Solution*: Future iterations will explore **RAG pipelines**, which retrieve only the most relevant chunks of information before generating a response  making the system scalable and accurate even with vast data.


<font size=6 color="#4682B4">Power Ahead!</font>
___